# 2. Sentiment Classification Using RNN

### Task:

Sentiment analysis determines if a given text expresses a positive or negative emotion. You will train an LSTM-based sentiment classifier using the IMDB dataset.

This task implements an LSTM-based recurrent neural network for sentiment classification. The model will learn patterns from movie reviews and classify each review as having either positive or negative sentiment.

In [6]:
# Import TensorFlow for loading the IMDB dataset and building the LSTM sentiment classification model.
import tensorflow as tf

# 1. Load the IMDB sentiment dataset (tensorflow.keras.datasets.imdb)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=10000)

print("Number of training reviews:", len(x_train))
print("Number of testing reviews:", len(x_test))

print("\nFirst review:")
print(x_train[0])

print("\nFirst review label:", y_train[0])

Number of training reviews: 25000
Number of testing reviews: 25000

First review:
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 1

In [7]:
# 2. Preprocess the text data by tokenization and padding sequences

# The Keras IMDB dataset is already tokenized in 1 above: each review was loaded as a sequence of integer word IDs.
# Therefore, no additional tokenization is required.

# Pad/truncate all reviews to the same length.
max_length = 200

x_train = tf.keras.utils.pad_sequences(
    x_train,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

x_test = tf.keras.utils.pad_sequences(
    x_test,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

# Verify the preprocessing.
print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)
print("\nLength of first review:", len(x_train[0]))
print("\nFirst padded review:")
print(x_train[0])

Training data shape: (25000, 200)
Testing data shape: (25000, 200)

Length of first review: 200

First padded review:
[   1   14   22   16   43  530  973 1622 1385   65  458 4468   66 3941
    4  173   36  256    5   25  100   43  838  112   50  670    2    9
   35  480  284    5  150    4  172  112  167    2  336  385   39    4
  172 4536 1111   17  546   38   13  447    4  192   50   16    6  147
 2025   19   14   22    4 1920 4613  469    4   22   71   87   12   16
   43  530   38   76   15   13 1247    4   22   17  515   17   12   16
  626   18    2    5   62  386   12    8  316    8  106    5    4 2223
 5244   16  480   66 3785   33    4  130   12   16   38  619    5   25
  124   51   36  135   48   25 1415   33    6   22   12  215   28   77
   52    5   14  407   16   82    2    8    4  107  117 5952   15  256
    4    2    7 3766    5  723   36   71   43  530  476   26  400  317
   46    7    4    2 1029   13  104   88    4  381   15  297   98   32
 2071   56   26  141    6  194

In [9]:
# 3. Train an LSTM-based model to classify reviews as positive or negative.

# Build the sentiment classification model.
model = tf.keras.Sequential([

    # Convert each word ID into a learned numerical representation that the LSTM can use.
    tf.keras.layers.Embedding(
        input_dim=10000,
        output_dim=128
    ),

    # Read the sequence of words in each review and learn patterns that can help distinguish positive reviews from negative reviews.
    tf.keras.layers.LSTM(64),

    # Produce ONE sentiment probability between 0 and 1.
    # A value below 0.5 is classified as Negative (0), while a value of 0.5 or above is classified as Positive (1).
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# Configure the model for binary classification because the IMDB labels contain only two classes:
# 0 = Negative and 1 = Positive.
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train the model using the reviews (x_train) and their known positive/negative labels (y_train).
# 20% of the training data is used to check validation performance.
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 90ms/step - accuracy: 0.5414 - loss: 0.6822 - val_accuracy: 0.5694 - val_loss: 0.6738
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 29s 94ms/step - accuracy: 0.5986 - loss: 0.6616 - val_accuracy: 0.6938 - val_loss: 0.6495
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 28s 91ms/step - accuracy: 0.7027 - loss: 0.5878 - val_accuracy: 0.7200 - val_loss: 0.5770
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 28s 91ms/step - accuracy: 0.7610 - loss: 0.4897 - val_accuracy: 0.8458 - val_loss: 0.3915
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 86ms/step - accuracy: 0.8975 - loss: 0.2612 - val_accuracy: 0.8658 - val_loss: 0.3365


In [10]:
# 4. Generate a confusion matrix and classification report (accuracy, precision, recall, F1-score).

# Import the evaluation tools needed to measure how well the trained
# LSTM distinguishes Negative (0) reviews from Positive (1) reviews.
from sklearn.metrics import confusion_matrix, classification_report

# Use the trained LSTM model to predict the sentiment of all test reviews.
# The sigmoid output gives a probability between 0 and 1 for each review.
y_prob = model.predict(x_test)

# Convert the predicted probabilities into the two sentiment classes:
# probability below 0.5  = Negative review (0)
# probability 0.5 or above = Positive review (1)
y_pred = (y_prob >= 0.5).astype("int32").flatten()

# Create the confusion matrix by comparing the model's predictions with the actual positive/negative labels in the test dataset.
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

# Generate the classification report.
# This shows precision, recall, F1-score, and accuracy for the model's Positive and Negative sentiment classifications.
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Negative", "Positive"]
    )
)

782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step
Confusion Matrix:
[[11085  1415]
 [ 2456 10044]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.82      0.89      0.85     12500
    Positive       0.88      0.80      0.84     12500

    accuracy                           0.85     25000
   macro avg       0.85      0.85      0.84     25000
weighted avg       0.85      0.85      0.84     25000



### 5. Precision-Recall Tradeoff

The precision-recall tradeoff is important because a sentiment classifier should not only make accurate positive or negative predictions, but should also balance **how reliable those predictions are** with **how many actual reviews it successfully identifies**.

**Precision** measures how often the model is correct when it predicts a particular sentiment, while **recall** measures how many reviews of that actual sentiment the model successfully identifies.

For example, the model achieved **0.88 precision** and **0.80 recall** for positive reviews. This means its positive predictions were generally reliable, but it still missed some actual positive reviews. The confusion matrix showed that **2,456 positive reviews were incorrectly classified as negative**.

Increasing recall may help the model identify more actual positive reviews, but it can also increase false positive predictions and reduce precision. Therefore, the appropriate balance between precision and recall depends on whether avoiding incorrect sentiment predictions or identifying as many relevant sentiments as possible is more important for the application.


### Observation

The IMDB dataset contained **25,000 training and 25,000 testing movie reviews**, with each review already tokenized into integer word IDs. The reviews were padded or truncated to **200 tokens** so they had a consistent input length for the LSTM model.

The LSTM was trained to classify reviews as **negative (0) or positive (1)**. During training, accuracy increased from **54.14% to 89.75%**, while validation accuracy increased from **56.94% to 86.58%**, showing that the model progressively learned useful sentiment patterns from the reviews.

On the unseen test data, the model achieved an overall accuracy of approximately **85%**. The confusion matrix showed that it correctly classified **11,085 negative reviews** and **10,044 positive reviews**, while misclassifying **1,415 negative reviews as positive** and **2,456 positive reviews as negative**. The classification report also showed that positive reviews had **88% precision but 80% recall**, demonstrating that a model can be reliable when making a prediction while still missing some reviews that belong to that class.

Overall, the experiment demonstrated how an LSTM can learn sequential patterns in text for sentiment classification and why **accuracy alone is not enough** to evaluate a classifier. Precision, recall, F1-score, and the confusion matrix provide a clearer understanding of the types of predictions the model gets right and wrong.
